# This method attempts to predict weight by matching a output label from the pretrained ResNet-50 to a value in the weight map

In [26]:
import torch
import torchvision.models as models
from typing import Dict, List

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


# Model initialization

In [27]:
# Initialize ResNet-50 model
resnet50 = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

# Move model to device
resnet50 = resnet50.to(device)
print(resnet50)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

# Weight map - maps class/label to a given object weight

In [28]:
# Maps class labels to their corresponding weights
# map is very incomplete right now i will fix it later
from typing import Dict
class_weight_map: Dict[str, float] = {
    
    # --- Birds & Small Animals ---
    'cock': 5.5,
    'hen': 4.0,
    'ostrich': 230.0,
    'bald_eagle': 10.0,
    'hummingbird': 0.01,
    'goldfish': 0.25,
    'tench': 4.5,
    'great white shark': 2400.0,
    
    # --- Reptiles ---
    'American alligator': 500.0,
    'Komodo dragon': 150.0,
    'loggerhead': 300.0,
    
    # --- Dogs ---
    'Chihuahua': 4.5,
    'beagle': 25.0,
    'Saint Bernard': 160.0,
    'Great Dane': 140.0,
    
    # --- Large Mammals ---
    'African elephant': 12000.0,
    'giant panda': 250.0,
    'brown bear': 600.0,
    'hippopotamus': 3500.0,
    'killer whale': 8000.0,
    
    # --- Vehicles & Large Structures ---
    'airliner': 180000.0,
    'aircraft carrier': 2e8,
    'school bus': 25000.0,
    'pickup': 5000.0,
    'mountain bike': 30.0,
    
    # --- Household Items ---
    'acoustic_guitar': 5.0,
    'backpack': 3.0,
    'laptop': 4.0,
    'microwave': 35.0,
    'beer bottle': 1.0,
    'dining table': 100.0,
    
    # --- Food ---
    'cheeseburger': 0.5,
    'granny smith': 0.4,
    'pizza': 2.0,
    'bagel': 0.2,
    
    # --- Insects ---
    'ant': 0.00001,
    'bee': 0.0002,
    'monarch': 0.001,
}

# Preprocess images and download labels

In [29]:
from PIL import Image
import torchvision.transforms as transforms
import urllib.request

# Image preprocessing pipeline, matches ImageNet normalization
image_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Download ImageNet class labels
try:
    with urllib.request.urlopen('https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt') as url:
        imagenet_classes = [line.decode('utf-8').strip() for line in url.readlines()]
    print(f"Loaded {len(imagenet_classes)} ImageNet classes")
except Exception as e:
    print(f"Warning: Could not download ImageNet classes: {e}")
    imagenet_classes = None

Loaded 1000 ImageNet classes


# Test inference function

In [30]:
def test_inference(image_path: str):
    """Test ResNet-50 inference on an image and return top prediction with weight"""
    try:
        image = Image.open(image_path).convert('RGB')
        image_tensor = image_transforms(image).unsqueeze(0).to(device)
        
        resnet50.eval()
        with torch.no_grad():
            outputs = resnet50(image_tensor)
            probabilities = torch.nn.functional.softmax(outputs, dim=1)
            top_prob, top_indices = torch.topk(probabilities, k=1)
        
        idx = top_indices[0].item()
        confidence = top_prob[0].item()
        class_name = imagenet_classes[idx].split(',')[0] if imagenet_classes else f"class_{idx}"
        weight = class_weight_map.get(class_name, 999999.99)
        
        return {
            'class': class_name,
            'confidence': f"{confidence:.4f}",
            'weight': weight
        }
    
    except FileNotFoundError:
        print(f"Error: Image file not found at {image_path}")
        return None
    except Exception as e:
        print(f"Error during inference: {e}")
        return None

# Uplaod image from URL and test inference

In [31]:
def test_inference_from_url(url: str):
    """Test inference on image from URL"""
    import urllib.request
    
    req = urllib.request.Request(url)
    
    try:
        with urllib.request.urlopen(req) as response:
            with open('temp_image.jpg', 'wb') as out_file:
                out_file.write(response.read())
        return test_inference('temp_image.jpg')
    except Exception as e:
        print(f"Error downloading image: {e}")
        return None

# test_inference_from_url('https://github.com/EliSchwartz/imagenet-sample-images/blob/master/n01443537_goldfish.JPEG?raw=true')
# test_inference_from_url('https://t3.ftcdn.net/jpg/01/74/93/80/360_F_174938002_zvgqpU18283OpwpHCA1hrfItZ76sMuMB.jpg')
test_inference_from_url('https://i.natgeofe.com/k/41fa020b-b01d-4fd6-9e08-5e072a8ba256/komodo-dragon-white-background.jpg?wp=1&w=1084.125&h=609')

{'class': 'Komodo dragon', 'confidence': '0.9816', 'weight': 150.0}